In [7]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Ready")

Ready


In [8]:
import requests

url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/prc_hpi_q"

params = {
    "geo": ["IE", "ES"],
    "purchase": "TOTAL",
    "unit": "I15_Q"
}
response = requests.get(url, params=params)

print("Status:", response.status_code)

Status: 200


In [9]:
data = response.json()

print(data.keys())


dict_keys(['version', 'class', 'label', 'source', 'updated', 'value', 'status', 'id', 'size', 'dimension', 'extension'])


In [10]:
print(data["label"])

House price index - quarterly data


In [11]:
data["dimension"].keys()


dict_keys(['freq', 'purchase', 'unit', 'geo', 'time'])

In [12]:
data["dimension"]["geo"]["category"]["label"]

{'IE': 'Ireland', 'ES': 'Spain'}

In [13]:
data["dimension"]["time"]["category"]["index"]

{'2005-Q1': 0,
 '2005-Q2': 1,
 '2005-Q3': 2,
 '2005-Q4': 3,
 '2006-Q1': 4,
 '2006-Q2': 5,
 '2006-Q3': 6,
 '2006-Q4': 7,
 '2007-Q1': 8,
 '2007-Q2': 9,
 '2007-Q3': 10,
 '2007-Q4': 11,
 '2008-Q1': 12,
 '2008-Q2': 13,
 '2008-Q3': 14,
 '2008-Q4': 15,
 '2009-Q1': 16,
 '2009-Q2': 17,
 '2009-Q3': 18,
 '2009-Q4': 19,
 '2010-Q1': 20,
 '2010-Q2': 21,
 '2010-Q3': 22,
 '2010-Q4': 23,
 '2011-Q1': 24,
 '2011-Q2': 25,
 '2011-Q3': 26,
 '2011-Q4': 27,
 '2012-Q1': 28,
 '2012-Q2': 29,
 '2012-Q3': 30,
 '2012-Q4': 31,
 '2013-Q1': 32,
 '2013-Q2': 33,
 '2013-Q3': 34,
 '2013-Q4': 35,
 '2014-Q1': 36,
 '2014-Q2': 37,
 '2014-Q3': 38,
 '2014-Q4': 39,
 '2015-Q1': 40,
 '2015-Q2': 41,
 '2015-Q3': 42,
 '2015-Q4': 43,
 '2016-Q1': 44,
 '2016-Q2': 45,
 '2016-Q3': 46,
 '2016-Q4': 47,
 '2017-Q1': 48,
 '2017-Q2': 49,
 '2017-Q3': 50,
 '2017-Q4': 51,
 '2018-Q1': 52,
 '2018-Q2': 53,
 '2018-Q3': 54,
 '2018-Q4': 55,
 '2019-Q1': 56,
 '2019-Q2': 57,
 '2019-Q3': 58,
 '2019-Q4': 59,
 '2020-Q1': 60,
 '2020-Q2': 61,
 '2020-Q3': 62,
 '

In [14]:
from itertools import product

dimension_ids = data["id"]
dimension_sizes = data["size"]

print("Dimensions:", dimension_ids)
print("Sizes:", dimension_sizes)

Dimensions: ['freq', 'purchase', 'unit', 'geo', 'time']
Sizes: [1, 1, 1, 2, 86]


In [15]:
dimension_values = {}

for dim in dimension_ids:
    index = data["dimension"][dim]["category"]["index"]
    
    if isinstance(index, dict):
        ordered = sorted(index.items(), key=lambda x: x[1])
        dimension_values[dim] = [item[0] for item in ordered]
    else:
        dimension_values[dim] = index

dimension_values

{'freq': ['Q'],
 'purchase': ['TOTAL'],
 'unit': ['I15_Q'],
 'geo': ['IE', 'ES'],
 'time': ['2005-Q1',
  '2005-Q2',
  '2005-Q3',
  '2005-Q4',
  '2006-Q1',
  '2006-Q2',
  '2006-Q3',
  '2006-Q4',
  '2007-Q1',
  '2007-Q2',
  '2007-Q3',
  '2007-Q4',
  '2008-Q1',
  '2008-Q2',
  '2008-Q3',
  '2008-Q4',
  '2009-Q1',
  '2009-Q2',
  '2009-Q3',
  '2009-Q4',
  '2010-Q1',
  '2010-Q2',
  '2010-Q3',
  '2010-Q4',
  '2011-Q1',
  '2011-Q2',
  '2011-Q3',
  '2011-Q4',
  '2012-Q1',
  '2012-Q2',
  '2012-Q3',
  '2012-Q4',
  '2013-Q1',
  '2013-Q2',
  '2013-Q3',
  '2013-Q4',
  '2014-Q1',
  '2014-Q2',
  '2014-Q3',
  '2014-Q4',
  '2015-Q1',
  '2015-Q2',
  '2015-Q3',
  '2015-Q4',
  '2016-Q1',
  '2016-Q2',
  '2016-Q3',
  '2016-Q4',
  '2017-Q1',
  '2017-Q2',
  '2017-Q3',
  '2017-Q4',
  '2018-Q1',
  '2018-Q2',
  '2018-Q3',
  '2018-Q4',
  '2019-Q1',
  '2019-Q2',
  '2019-Q3',
  '2019-Q4',
  '2020-Q1',
  '2020-Q2',
  '2020-Q3',
  '2020-Q4',
  '2021-Q1',
  '2021-Q2',
  '2021-Q3',
  '2021-Q4',
  '2022-Q1',
  '2022-Q2',


In [16]:
combinations = list(product(*[
    dimension_values[dim] for dim in dimension_ids
]))

rows = []

for position, combination in enumerate(combinations):
    value = data.get("value", {}).get(str(position))
    
    row = dict(zip(dimension_ids, combination))
    row["house_price_index"] = value
    
    rows.append(row)

hpi = pd.DataFrame(rows)

hpi.head()

,freq,purchase,unit,geo,time,house_price_index
0,Q,TOTAL,I15_Q,IE,2005-Q1,125.92
1,Q,TOTAL,I15_Q,IE,2005-Q2,128.58
2,Q,TOTAL,I15_Q,IE,2005-Q3,133.58
3,Q,TOTAL,I15_Q,IE,2005-Q4,139.46
4,Q,TOTAL,I15_Q,IE,2006-Q1,141.99


In [17]:
hpi.shape

(172, 6)

In [18]:
hpi.columns

Index(['freq', 'purchase', 'unit', 'geo', 'time', 'house_price_index'], dtype='str')

In [19]:
hpi.tail

<bound method NDFrame.tail of     freq purchase   unit geo     time  house_price_index
0      Q    TOTAL  I15_Q  IE  2005-Q1             125.92
1      Q    TOTAL  I15_Q  IE  2005-Q2             128.58
2      Q    TOTAL  I15_Q  IE  2005-Q3             133.58
3      Q    TOTAL  I15_Q  IE  2005-Q4             139.46
4      Q    TOTAL  I15_Q  IE  2006-Q1             141.99
..   ...      ...    ...  ..      ...                ...
167    Q    TOTAL  I15_Q  ES  2025-Q2             178.89
168    Q    TOTAL  I15_Q  ES  2025-Q3             184.06
169    Q    TOTAL  I15_Q  ES  2025-Q4             187.45
170    Q    TOTAL  I15_Q  ES  2026-Q1             194.06
171    Q    TOTAL  I15_Q  ES  2026-Q2             200.60

[172 rows x 6 columns]>

In [20]:
hpi = hpi[
    ["geo", "time", "house_price_index"]
].copy()

hpi["country"] = hpi["geo"].map({
    "IE": "Ireland",
    "ES": "Spain"
})

hpi = hpi[
    ["country", "time", "house_price_index"]
]

hpi.head()

,country,time,house_price_index
0,Ireland,2005-Q1,125.92
1,Ireland,2005-Q2,128.58
2,Ireland,2005-Q3,133.58
3,Ireland,2005-Q4,139.46
4,Ireland,2006-Q1,141.99


In [21]:
hpi.dropna().groupby("country").tail(8)

,country,time,house_price_index
78,Ireland,2024-Q3,185.50
79,Ireland,2024-Q4,189.97
80,Ireland,2025-Q1,191.83
81,Ireland,2025-Q2,194.34
82,Ireland,2025-Q3,199.36
83,Ireland,2025-Q4,203.23
84,Ireland,2026-Q1,204.94
85,Ireland,2026-Q2,205.94
164,Spain,2024-Q3,163.12
165,Spain,2024-Q4,166.07


In [22]:
# ---------------------------------------------------------
# CALCULATE YEAR-ON-YEAR HOUSE PRICE GROWTH
# ---------------------------------------------------------
#
# Eurostat gives us a House Price Index (HPI).
#
# An index tells us how house prices have changed relative
# to a base period, but the index itself isn't always very
# intuitive for an investor.
#
# We therefore calculate the percentage change compared
# with the SAME QUARTER one year earlier.
#
# Because our data is quarterly:
#     4 rows earlier = same quarter last year
#
# Example:
#     2025 Q1 HPI = 130
#     2024 Q1 HPI = 125
#
#     YoY growth = ((130 / 125) - 1) * 100
#                = 4%
#
# IMPORTANT:
# We calculate this in Python rather than asking the AI
# to calculate it. This keeps numerical analysis
# deterministic and reproducible.
# ---------------------------------------------------------


# Sort each country's observations chronologically.
# This is important because pct_change(4) assumes the rows
# are in the correct quarterly order.
hpi = hpi.sort_values(
    by=["country", "time"]
).reset_index(drop=True)


# Calculate percentage change from four quarters earlier.
#
# groupby("country") ensures that Ireland is compared only
# with previous Irish observations and Spain only with
# previous Spanish observations.
#
# fill_method=None prevents Pandas from automatically
# filling missing observations before calculating growth.
hpi["house_price_yoy_pct"] = (
    hpi
    .groupby("country")["house_price_index"]
    .pct_change(periods=4, fill_method=None)
    * 100
)


# Round to two decimal places to make the output easier
# to read in the dashboard.
hpi["house_price_yoy_pct"] = (
    hpi["house_price_yoy_pct"].round(2)
)


# Display the latest observations.
hpi.dropna().groupby("country").tail(8)

,country,time,house_price_index,house_price_yoy_pct
78,Ireland,2024-Q3,185.50,9.90
79,Ireland,2024-Q4,189.97,9.37
80,Ireland,2025-Q1,191.83,7.95
81,Ireland,2025-Q2,194.34,7.76
82,Ireland,2025-Q3,199.36,7.47
83,Ireland,2025-Q4,203.23,6.98
84,Ireland,2026-Q1,204.94,6.83
85,Ireland,2026-Q2,205.94,5.97
164,Spain,2024-Q3,163.12,8.26
165,Spain,2024-Q4,166.07,11.36


In [23]:
# Show the most recent available observation for each country.
#
# This is similar to what will eventually feed the KPI cards
# at the top of our Streamlit dashboard.

latest_hpi = (
    hpi
    .dropna(subset=["house_price_index"])
    .groupby("country")
    .tail(1)
)

latest_hpi

,country,time,house_price_index,house_price_yoy_pct
85,Ireland,2026-Q2,205.94,5.97
171,Spain,2026-Q2,200.60,12.14


In [24]:
# Create a simplified view containing only the fields that
# would be useful to the dashboard or AI analysis layer.

latest_hpi[
    [
        "country",
        "time",
        "house_price_index",
        "house_price_yoy_pct"
    ]
]

,country,time,house_price_index,house_price_yoy_pct
85,Ireland,2026-Q2,205.94,5.97
171,Spain,2026-Q2,200.60,12.14


In [25]:
# ---------------------------------------------------------
# VALIDATE THE HOUSE PRICE DATA
# ---------------------------------------------------------
#
# Instead of plotting the data in Jupyter, we'll inspect
# the latest observations directly.
#
# Our final visualisations will be created in Streamlit,
# so we don't need to install Matplotlib just for the
# notebook.
# ---------------------------------------------------------

# Show the latest 8 quarters for each country.
latest_periods = (
    hpi
    .dropna(subset=["house_price_index"])
    .groupby("country")
    .tail(8)
)

latest_periods

,country,time,house_price_index,house_price_yoy_pct
78,Ireland,2024-Q3,185.50,9.90
79,Ireland,2024-Q4,189.97,9.37
80,Ireland,2025-Q1,191.83,7.95
81,Ireland,2025-Q2,194.34,7.76
82,Ireland,2025-Q3,199.36,7.47
83,Ireland,2025-Q4,203.23,6.98
84,Ireland,2026-Q1,204.94,6.83
85,Ireland,2026-Q2,205.94,5.97
164,Spain,2024-Q3,163.12,8.26
165,Spain,2024-Q4,166.07,11.36


In [26]:
# Create a comparison table.
#
# Each row represents a quarter.
# Each column represents a country.
#
# This makes it easy to visually compare the House Price
# Index between Ireland and Spain.

hpi_comparison = hpi.pivot(
    index="time",
    columns="country",
    values="house_price_index"
)

hpi_comparison.tail(12)

country,Ireland,Spain
time,,
2023-Q3,168.79,150.68
2023-Q4,173.69,149.13
2024-Q1,177.70,153.18
2024-Q2,180.34,158.63
2024-Q3,185.50,163.12
2024-Q4,189.97,166.07
2025-Q1,191.83,172.00
2025-Q2,194.34,178.89
2025-Q3,199.36,184.06


In [27]:
# Create the same comparison table using annual
# house-price growth.
#
# This will eventually become one of the main metrics
# displayed in our Streamlit dashboard.

growth_comparison = hpi.pivot(
    index="time",
    columns="country",
    values="house_price_yoy_pct"
)

growth_comparison.tail(12)

country,Ireland,Spain
time,,
2023-Q3,1.39,4.53
2023-Q4,3.14,4.29
2024-Q1,6.31,6.39
2024-Q2,8.44,7.93
2024-Q3,9.90,8.26
2024-Q4,9.37,11.36
2025-Q1,7.95,12.29
2025-Q2,7.76,12.77
2025-Q3,7.47,12.84


In [28]:
# ---------------------------------------------------------
# DOWNLOAD RENT INDEX DATA FROM EUROSTAT
# ---------------------------------------------------------
#
# We now want to understand how residential rents are
# changing alongside house prices.
#
# Eurostat publishes rent data as part of the Harmonised
# Index of Consumer Prices (HICP).
#
# We will:
#   1. Request rent data for Ireland and Spain
#   2. Convert the Eurostat JSON response into a DataFrame
#   3. Calculate annual rental growth
#   4. Eventually combine this with our house-price data
#
# IMPORTANT:
# We continue using official source data rather than
# asking the AI for these figures.
# ---------------------------------------------------------

import requests

rent_url = (
    "https://ec.europa.eu/eurostat/api/dissemination/"
    "statistics/1.0/data/prc_hicp_midx"
)

rent_params = {
    # Ireland and Spain
    "geo": ["IE", "ES"],

    # Actual rentals for housing
    "coicop": "CP041",

    # Index with 2015 = 100
    "unit": "I15"
}

rent_response = requests.get(
    rent_url,
    params=rent_params,
    timeout=30
)

print("Status:", rent_response.status_code)

Status: 200


In [29]:
# Convert the API response from JSON into a Python
# dictionary so that we can inspect its structure.

rent_data = rent_response.json()


# Display the name/description of the dataset.
print(rent_data["label"])


# Display the dimensions contained in the dataset.
#
# We expect dimensions such as:
#   geo     -> country
#   time    -> month
#   coicop  -> type of consumer expenditure
#   unit    -> measurement unit

print(rent_data["id"])
print(rent_data["size"])

HICP - monthly data (index) (1996-2025)
['freq', 'unit', 'coicop', 'geo', 'time']
[1, 1, 1, 2, 360]


In [30]:
# Check which countries were returned by Eurostat.

rent_data["dimension"]["geo"]["category"]["label"]

{'IE': 'Ireland', 'ES': 'Spain'}

In [31]:
# ---------------------------------------------------------
# CONVERT EUROSTAT JSON-STAT DATA INTO A DATAFRAME
# ---------------------------------------------------------
#
# Eurostat stores observations as a multidimensional
# statistical dataset rather than a simple table.
#
# We therefore:
#
#   1. Determine the possible values for every dimension
#   2. Generate every combination of those dimensions
#   3. Match each combination to its observation value
#   4. Convert everything into a Pandas DataFrame
#
# Later we could turn this into a reusable function, but
# for now we're keeping the process visible so that it's
# easy to understand.
# ---------------------------------------------------------

from itertools import product


# Get the dimension names used by this dataset.
rent_dimension_ids = rent_data["id"]


# Create a dictionary that will contain the ordered values
# belonging to each dimension.

rent_dimension_values = {}


for dim in rent_dimension_ids:

    # Eurostat provides an index showing the position of
    # each category within a dimension.
    index = rent_data["dimension"][dim]["category"]["index"]

    if isinstance(index, dict):

        # Sort categories by their numerical position.
        ordered = sorted(
            index.items(),
            key=lambda x: x[1]
        )

        rent_dimension_values[dim] = [
            item[0] for item in ordered
        ]

    else:
        rent_dimension_values[dim] = index


rent_dimension_values

{'freq': ['M'],
 'unit': ['I15'],
 'coicop': ['CP041'],
 'geo': ['IE', 'ES'],
 'time': ['1996-01',
  '1996-02',
  '1996-03',
  '1996-04',
  '1996-05',
  '1996-06',
  '1996-07',
  '1996-08',
  '1996-09',
  '1996-10',
  '1996-11',
  '1996-12',
  '1997-01',
  '1997-02',
  '1997-03',
  '1997-04',
  '1997-05',
  '1997-06',
  '1997-07',
  '1997-08',
  '1997-09',
  '1997-10',
  '1997-11',
  '1997-12',
  '1998-01',
  '1998-02',
  '1998-03',
  '1998-04',
  '1998-05',
  '1998-06',
  '1998-07',
  '1998-08',
  '1998-09',
  '1998-10',
  '1998-11',
  '1998-12',
  '1999-01',
  '1999-02',
  '1999-03',
  '1999-04',
  '1999-05',
  '1999-06',
  '1999-07',
  '1999-08',
  '1999-09',
  '1999-10',
  '1999-11',
  '1999-12',
  '2000-01',
  '2000-02',
  '2000-03',
  '2000-04',
  '2000-05',
  '2000-06',
  '2000-07',
  '2000-08',
  '2000-09',
  '2000-10',
  '2000-11',
  '2000-12',
  '2001-01',
  '2001-02',
  '2001-03',
  '2001-04',
  '2001-05',
  '2001-06',
  '2001-07',
  '2001-08',
  '2001-09',
  '2001-10',
  '2

In [32]:
# Generate every possible combination of the dimensions.
#
# For example:
#
# Ireland | Rent | Index | January 2024
# Ireland | Rent | Index | February 2024
# Spain   | Rent | Index | January 2024
# etc.

rent_combinations = list(
    product(
        *[
            rent_dimension_values[dim]
            for dim in rent_dimension_ids
        ]
    )
)


# Create one row for every observation.

rent_rows = []

for position, combination in enumerate(rent_combinations):

    # Retrieve the statistical value associated with this
    # position in the Eurostat dataset.
    value = rent_data.get(
        "value",
        {}
    ).get(str(position))

    # Convert the dimension combination into column/value
    # pairs.
    row = dict(
        zip(
            rent_dimension_ids,
            combination
        )
    )

    # Add the actual rent index observation.
    row["rent_index"] = value

    rent_rows.append(row)


# Convert our list of observations into a DataFrame.

rent = pd.DataFrame(rent_rows)

rent.head()

,freq,unit,coicop,geo,time,rent_index
0,M,I15,CP041,IE,1996-01,51.8
1,M,I15,CP041,IE,1996-02,52.0
2,M,I15,CP041,IE,1996-03,52.3
3,M,I15,CP041,IE,1996-04,52.4
4,M,I15,CP041,IE,1996-05,52.6


In [33]:
# ---------------------------------------------------------
# CLEAN THE RENT DATA
# ---------------------------------------------------------
#
# The raw Eurostat dataset contains technical fields that
# we don't need in our analytical model.
#
# For the application we only need:
#
#   country
#   time
#   rent_index
# ---------------------------------------------------------


# Keep only the relevant columns.

rent = rent[
    [
        "geo",
        "time",
        "rent_index"
    ]
].copy()


# Replace Eurostat country codes with readable names.

rent["country"] = rent["geo"].map({
    "IE": "Ireland",
    "ES": "Spain"
})


# Remove the technical geo code now that we have the
# readable country name.

rent = rent[
    [
        "country",
        "time",
        "rent_index"
    ]
]


# Look at the latest observations.

rent.dropna().groupby("country").tail(12)

,country,time,rent_index
348,Ireland,2025-01,171.10
349,Ireland,2025-02,171.30
350,Ireland,2025-03,172.00
351,Ireland,2025-04,172.20
352,Ireland,2025-05,172.80
353,Ireland,2025-06,173.40
354,Ireland,2025-07,173.60
355,Ireland,2025-08,174.20
356,Ireland,2025-09,175.00
357,Ireland,2025-10,175.30


In [34]:
# ---------------------------------------------------------
# CONVERT MONTHLY RENT DATA TO QUARTERLY DATA
# ---------------------------------------------------------
#
# Our House Price Index is quarterly.
#
# To combine rents with house prices, we need both datasets
# to use the same time frequency.
#
# We will therefore convert each monthly rent observation
# into a calendar quarter and calculate the average rent
# index for that quarter.
#
# Example:
#
# January  120
# February 121
# March    122
#
# Q1 average = 121
# ---------------------------------------------------------


# Convert Eurostat's time field into a proper Pandas
# datetime value.

rent["date"] = pd.to_datetime(
    rent["time"]
)


# Convert each month into its corresponding quarter.
#
# Example:
#   2025-01 -> 2025Q1
#   2025-02 -> 2025Q1
#   2025-03 -> 2025Q1
#   2025-04 -> 2025Q2

rent["quarter"] = (
    rent["date"]
    .dt
    .to_period("Q")
)


# Calculate the average rent index within each quarter.

rent_quarterly = (
    rent
    .groupby(
        ["country", "quarter"],
        as_index=False
    )
    ["rent_index"]
    .mean()
)


# Round the index for readability.

rent_quarterly["rent_index"] = (
    rent_quarterly["rent_index"]
    .round(2)
)


rent_quarterly.tail(10)

,country,quarter,rent_index
230,Spain,2023Q3,108.82
231,Spain,2023Q4,109.27
232,Spain,2024Q1,109.91
233,Spain,2024Q2,110.72
234,Spain,2024Q3,111.24
235,Spain,2024Q4,111.86
236,Spain,2025Q1,112.53
237,Spain,2025Q2,113.37
238,Spain,2025Q3,113.95
239,Spain,2025Q4,114.58


In [35]:
# ---------------------------------------------------------
# CALCULATE YEAR-ON-YEAR RENTAL GROWTH
# ---------------------------------------------------------
#
# Because the data is now quarterly:
#
#   4 observations earlier = same quarter last year
#
# We calculate:
#
#   ((current rent index / previous-year rent index) - 1)
#   × 100
#
# Again, this calculation happens in Python rather than
# inside the AI model.
# ---------------------------------------------------------


# Make sure observations are in chronological order.

rent_quarterly = (
    rent_quarterly
    .sort_values(
        ["country", "quarter"]
    )
    .reset_index(drop=True)
)


# Calculate annual rental growth.

rent_quarterly["rent_yoy_pct"] = (
    rent_quarterly
    .groupby("country")["rent_index"]
    .pct_change(
        periods=4,
        fill_method=None
    )
    * 100
)


# Round the result.

rent_quarterly["rent_yoy_pct"] = (
    rent_quarterly["rent_yoy_pct"]
    .round(2)
)


# Show the latest eight quarters for each market.

rent_quarterly.dropna().groupby("country").tail(8)

,country,quarter,rent_index,rent_yoy_pct
112,Ireland,2024Q1,162.93,5.82
113,Ireland,2024Q2,164.33,5.29
114,Ireland,2024Q3,167.07,4.79
115,Ireland,2024Q4,169.87,5.18
116,Ireland,2025Q1,171.47,5.24
117,Ireland,2025Q2,172.80,5.15
118,Ireland,2025Q3,174.27,4.31
119,Ireland,2025Q4,175.80,3.49
232,Spain,2024Q1,109.91,2.02
233,Spain,2024Q2,110.72,2.17


In [36]:
# ---------------------------------------------------------
# CALCULATE YEAR-ON-YEAR RENTAL GROWTH
# ---------------------------------------------------------
#
# Because the data is now quarterly:
#
#   4 observations earlier = same quarter last year
#
# We calculate:
#
#   ((current rent index / previous-year rent index) - 1)
#   × 100
#
# Again, this calculation happens in Python rather than
# inside the AI model.
# ---------------------------------------------------------


# Make sure observations are in chronological order.

rent_quarterly = (
    rent_quarterly
    .sort_values(
        ["country", "quarter"]
    )
    .reset_index(drop=True)
)


# Calculate annual rental growth.

rent_quarterly["rent_yoy_pct"] = (
    rent_quarterly
    .groupby("country")["rent_index"]
    .pct_change(
        periods=4,
        fill_method=None
    )
    * 100
)


# Round the result.

rent_quarterly["rent_yoy_pct"] = (
    rent_quarterly["rent_yoy_pct"]
    .round(2)
)


# Show the latest eight quarters for each market.

rent_quarterly.dropna().groupby("country").tail(8)

,country,quarter,rent_index,rent_yoy_pct
112,Ireland,2024Q1,162.93,5.82
113,Ireland,2024Q2,164.33,5.29
114,Ireland,2024Q3,167.07,4.79
115,Ireland,2024Q4,169.87,5.18
116,Ireland,2025Q1,171.47,5.24
117,Ireland,2025Q2,172.80,5.15
118,Ireland,2025Q3,174.27,4.31
119,Ireland,2025Q4,175.80,3.49
232,Spain,2024Q1,109.91,2.02
233,Spain,2024Q2,110.72,2.17


In [37]:
# ---------------------------------------------------------
# STANDARDISE THE TIME FIELD
# ---------------------------------------------------------
#
# Our two datasets currently use slightly different names:
#
# House prices:  time
# Rent data:     quarter
#
# Both represent calendar quarters, so we'll convert them
# into the same Pandas quarterly-period format.
#
# Having a consistent time field is essential before we
# merge datasets.
# ---------------------------------------------------------


# Convert the Eurostat house-price time field into a
# quarterly Period.
#
# Example:
#     "2025-Q1" -> Period("2025Q1")

hpi["quarter"] = pd.PeriodIndex(
    hpi["time"],
    freq="Q"
)


# We no longer need the original "time" column.

hpi = hpi.drop(
    columns=["time"]
)


# Check the result.

hpi.tail()

,country,house_price_index,house_price_yoy_pct,quarter
167,Spain,178.89,12.77,2025Q2
168,Spain,184.06,12.84,2025Q3
169,Spain,187.45,12.87,2025Q4
170,Spain,194.06,12.83,2026Q1
171,Spain,200.60,12.14,2026Q2


In [38]:
# ---------------------------------------------------------
# MERGE HOUSE PRICE AND RENT DATA
# ---------------------------------------------------------
#
# Both datasets now share two identifiers:
#
#     country
#     quarter
#
# We can therefore join the observations together.
#
# We use an OUTER join rather than an INNER join.
#
# Why?
#
# An inner join would silently remove quarters where one
# dataset is missing.
#
# An outer join keeps all available observations and makes
# missing data visible as NaN.
#
# This is safer for analytical work because we don't want
# missing data to disappear without us noticing.
# ---------------------------------------------------------


market_data = pd.merge(
    hpi,
    rent_quarterly,
    on=["country", "quarter"],
    how="outer"
)


# Sort the finished table chronologically.

market_data = (
    market_data
    .sort_values(
        ["country", "quarter"]
    )
    .reset_index(drop=True)
)


market_data.tail(10)

,country,house_price_index,house_price_yoy_pct,quarter,rent_index,rent_yoy_pct
234,Spain,153.18,6.39,2024Q1,109.91,2.02
235,Spain,158.63,7.93,2024Q2,110.72,2.17
236,Spain,163.12,8.26,2024Q3,111.24,2.22
237,Spain,166.07,11.36,2024Q4,111.86,2.37
238,Spain,172.00,12.29,2025Q1,112.53,2.38
239,Spain,178.89,12.77,2025Q2,113.37,2.39
240,Spain,184.06,12.84,2025Q3,113.95,2.44
241,Spain,187.45,12.87,2025Q4,114.58,2.43
242,Spain,194.06,12.83,2026Q1,NaN,NaN
243,Spain,200.60,12.14,2026Q2,NaN,NaN


In [39]:
# ---------------------------------------------------------
# DATA QUALITY CHECK
# ---------------------------------------------------------
#
# Look at the latest eight quarters for each country.
#
# We specifically want to check whether house-price and
# rental observations appear on the same rows.
# ---------------------------------------------------------


latest_market_data = (
    market_data
    .groupby("country")
    .tail(8)
)


latest_market_data

,country,house_price_index,house_price_yoy_pct,quarter,rent_index,rent_yoy_pct
114,Ireland,185.50,9.90,2024Q3,167.07,4.79
115,Ireland,189.97,9.37,2024Q4,169.87,5.18
116,Ireland,191.83,7.95,2025Q1,171.47,5.24
117,Ireland,194.34,7.76,2025Q2,172.80,5.15
118,Ireland,199.36,7.47,2025Q3,174.27,4.31
119,Ireland,203.23,6.98,2025Q4,175.80,3.49
120,Ireland,204.94,6.83,2026Q1,NaN,NaN
121,Ireland,205.94,5.97,2026Q2,NaN,NaN
236,Spain,163.12,8.26,2024Q3,111.24,2.22
237,Spain,166.07,11.36,2024Q4,111.86,2.37


In [40]:
# ---------------------------------------------------------
# CALCULATE PRICE / RENT DIVERGENCE
# ---------------------------------------------------------
#
# This measures the difference between annual house-price
# growth and annual rental growth.
#
# Formula:
#
#     House Price YoY % - Rent YoY %
#
#
# Example:
#
#     House prices       +8%
#     Rents              +4%
#
#     Divergence         +4 percentage points
#
#
# A positive value means:
#
#     property prices are growing faster than rents.
#
#
# A negative value means:
#
#     rents are growing faster than property prices.
#
#
# IMPORTANT:
#
# This is NOT automatically a "risk score".
#
# It is simply an analytical indicator. The meaning must
# be interpreted alongside financing conditions, inflation,
# economic growth and other evidence.
# ---------------------------------------------------------


market_data["price_rent_divergence_pp"] = (
    market_data["house_price_yoy_pct"]
    - market_data["rent_yoy_pct"]
)


market_data["price_rent_divergence_pp"] = (
    market_data["price_rent_divergence_pp"]
    .round(2)
)


market_data[
    [
        "country",
        "quarter",
        "house_price_yoy_pct",
        "rent_yoy_pct",
        "price_rent_divergence_pp"
    ]
].dropna().groupby("country").tail(8)

,country,quarter,house_price_yoy_pct,rent_yoy_pct,price_rent_divergence_pp
112,Ireland,2024Q1,6.31,5.82,0.49
113,Ireland,2024Q2,8.44,5.29,3.15
114,Ireland,2024Q3,9.90,4.79,5.11
115,Ireland,2024Q4,9.37,5.18,4.19
116,Ireland,2025Q1,7.95,5.24,2.71
117,Ireland,2025Q2,7.76,5.15,2.61
118,Ireland,2025Q3,7.47,4.31,3.16
119,Ireland,2025Q4,6.98,3.49,3.49
234,Spain,2024Q1,6.39,2.02,4.37
235,Spain,2024Q2,7.93,2.17,5.76


In [41]:
# ---------------------------------------------------------
# FIND THE LATEST COMPLETE PROPERTY-MARKET OBSERVATION
# ---------------------------------------------------------
#
# We only want quarters where BOTH house-price growth and
# rental growth are available.
#
# This prevents the dashboard from presenting an incomplete
# comparison as if it were complete.
# ---------------------------------------------------------


complete_property_data = market_data.dropna(
    subset=[
        "house_price_yoy_pct",
        "rent_yoy_pct"
    ]
)


latest_property_data = (
    complete_property_data
    .groupby("country")
    .tail(1)
)


latest_property_data[
    [
        "country",
        "quarter",
        "house_price_yoy_pct",
        "rent_yoy_pct",
        "price_rent_divergence_pp"
    ]
]

,country,quarter,house_price_yoy_pct,rent_yoy_pct,price_rent_divergence_pp
119,Ireland,2025Q4,6.98,3.49,3.49
241,Spain,2025Q4,12.87,2.43,10.44


In [42]:
# ---------------------------------------------------------
# DATA QUALITY SUMMARY
# ---------------------------------------------------------
#
# Count missing values in every column.
#
# This allows us to identify where source datasets have
# different publication schedules or incomplete history.
# ---------------------------------------------------------


market_data.isna().sum()

country                      0
house_price_index           75
house_price_yoy_pct         83
quarter                      0
rent_index                   4
rent_yoy_pct                12
price_rent_divergence_pp    87
dtype: int64

In [43]:
# Count the number of observations available for each
# country.

market_data.groupby("country").size()

country
Ireland    122
Spain      122
dtype: int64

In [44]:
# Show the earliest and latest quarter available for each
# market.

market_data.groupby("country")["quarter"].agg(
    ["min", "max"]
)

,min,max
country,,
Ireland,1996Q1,2026Q2
Spain,1996Q1,2026Q2


In [45]:
# ---------------------------------------------------------
# DOWNLOAD GENERAL INFLATION DATA
# ---------------------------------------------------------
#
# Source:
# Eurostat Harmonised Index of Consumer Prices (HICP)
#
# We use the overall consumer-price index rather than the
# rent component used previously.
#
# The data is monthly, so later we'll convert it to
# quarterly frequency.
# ---------------------------------------------------------


inflation_url = (
    "https://ec.europa.eu/eurostat/api/dissemination/"
    "statistics/1.0/data/prc_hicp_manr"
)


inflation_params = {

    # Ireland and Spain
    "geo": ["IE", "ES"],

    # All-items HICP
    "coicop": "CP00",

    # Annual rate of change
    "unit": "RCH_A"
}


inflation_response = requests.get(
    inflation_url,
    params=inflation_params,
    timeout=30
)


print(
    "Status:",
    inflation_response.status_code
)

Status: 200


In [46]:
# Convert the API response to a Python dictionary.

inflation_data = inflation_response.json()


# Check the dataset description and dimensions before
# processing it.

print(inflation_data["label"])
print(inflation_data["id"])
print(inflation_data["size"])

HICP - monthly data (annual rate of change) (1997-2025)
['freq', 'unit', 'coicop', 'geo', 'time']
[1, 1, 1, 2, 348]


In [47]:
# ---------------------------------------------------------
# CONVERT EUROSTAT INFLATION JSON INTO A DATAFRAME
# ---------------------------------------------------------
#
# This follows the same process we used for house prices
# and rents.
#
# Eurostat returns multidimensional JSON-stat data.
# We convert every observation into a normal table where
# each row represents:
#
#     Country | Month | Inflation Rate
#
# ---------------------------------------------------------


# Get the dimensions contained in the Eurostat dataset.
inflation_dimension_ids = inflation_data["id"]


# Create a dictionary containing the ordered values
# belonging to each dimension.
inflation_dimension_values = {}


for dim in inflation_dimension_ids:

    index = (
        inflation_data["dimension"][dim]
        ["category"]["index"]
    )

    if isinstance(index, dict):

        # Eurostat stores each category with a numerical
        # position. Sort by that position so our values
        # remain aligned with the observations.
        ordered = sorted(
            index.items(),
            key=lambda x: x[1]
        )

        inflation_dimension_values[dim] = [
            item[0] for item in ordered
        ]

    else:

        inflation_dimension_values[dim] = index


# Generate every possible combination of dimensions.
inflation_combinations = list(
    product(
        *[
            inflation_dimension_values[dim]
            for dim in inflation_dimension_ids
        ]
    )
)


# Build one row for every observation.
inflation_rows = []

for position, combination in enumerate(
    inflation_combinations
):

    value = inflation_data.get(
        "value",
        {}
    ).get(str(position))

    row = dict(
        zip(
            inflation_dimension_ids,
            combination
        )
    )

    row["inflation_pct"] = value

    inflation_rows.append(row)


# Convert to a Pandas DataFrame.
inflation = pd.DataFrame(inflation_rows)


inflation.head()

,freq,unit,coicop,geo,time,inflation_pct
0,M,RCH_A,CP00,IE,1997-01,1.8
1,M,RCH_A,CP00,IE,1997-02,1.7
2,M,RCH_A,CP00,IE,1997-03,1.3
3,M,RCH_A,CP00,IE,1997-04,1.6
4,M,RCH_A,CP00,IE,1997-05,1.5


In [48]:
# ---------------------------------------------------------
# CLEAN INFLATION DATA
# ---------------------------------------------------------
#
# Keep only the fields needed by our application:
#
#     country
#     time
#     inflation_pct
#
# inflation_pct is already the annual percentage change
# reported by Eurostat.
# ---------------------------------------------------------


inflation = inflation[
    [
        "geo",
        "time",
        "inflation_pct"
    ]
].copy()


# Convert country codes into readable labels.
inflation["country"] = inflation["geo"].map({
    "IE": "Ireland",
    "ES": "Spain"
})


# Remove the technical Eurostat geo field.
inflation = inflation[
    [
        "country",
        "time",
        "inflation_pct"
    ]
]


inflation.dropna().tail(10)

,country,time,inflation_pct
686,Spain,2025-03,2.2
687,Spain,2025-04,2.2
688,Spain,2025-05,2.0
689,Spain,2025-06,2.3
690,Spain,2025-07,2.7
691,Spain,2025-08,2.7
692,Spain,2025-09,3.0
693,Spain,2025-10,3.2
694,Spain,2025-11,3.2
695,Spain,2025-12,3.0


In [49]:
# ---------------------------------------------------------
# CONVERT MONTHLY INFLATION TO QUARTERLY
# ---------------------------------------------------------
#
# House prices are quarterly.
#
# Inflation is reported monthly.
#
# We therefore calculate the average annual inflation rate
# observed during each quarter.
#
# Example:
#
# January     2.0%
# February    2.2%
# March       2.4%
#
# Q1 average = 2.2%
# ---------------------------------------------------------


# Convert Eurostat's monthly time field into datetime.
inflation["date"] = pd.to_datetime(
    inflation["time"]
)


# Assign each month to a calendar quarter.
inflation["quarter"] = (
    inflation["date"]
    .dt
    .to_period("Q")
)


# Calculate average inflation during each quarter.
inflation_quarterly = (
    inflation
    .groupby(
        ["country", "quarter"],
        as_index=False
    )
    ["inflation_pct"]
    .mean()
)


# Round for readability.
inflation_quarterly["inflation_pct"] = (
    inflation_quarterly["inflation_pct"]
    .round(2)
)


inflation_quarterly.tail(10)

,country,quarter,inflation_pct
222,Spain,2023Q3,2.60
223,Spain,2023Q4,3.37
224,Spain,2024Q1,3.23
225,Spain,2024Q2,3.60
226,Spain,2024Q3,2.33
227,Spain,2024Q4,2.33
228,Spain,2025Q1,2.67
229,Spain,2025Q2,2.17
230,Spain,2025Q3,2.80
231,Spain,2025Q4,3.13


In [50]:
# ---------------------------------------------------------
# ADD INFLATION TO OUR MASTER MARKET DATASET
# ---------------------------------------------------------
#
# market_data currently contains:
#
#   House prices
#   Rental prices
#   Price/rent divergence
#
# We now add quarterly inflation.
#
# Again we use an OUTER join so that missing observations
# remain visible rather than being silently discarded.
# ---------------------------------------------------------


market_data = pd.merge(
    market_data,
    inflation_quarterly,
    on=["country", "quarter"],
    how="outer"
)


# Sort chronologically after the merge.
market_data = (
    market_data
    .sort_values(
        ["country", "quarter"]
    )
    .reset_index(drop=True)
)


market_data.groupby("country").tail(8)

,country,house_price_index,house_price_yoy_pct,quarter,rent_index,rent_yoy_pct,price_rent_divergence_pp,inflation_pct
114,Ireland,185.50,9.90,2024Q3,167.07,4.79,5.11,0.87
115,Ireland,189.97,9.37,2024Q4,169.87,5.18,4.19,0.53
116,Ireland,191.83,7.95,2025Q1,171.47,5.24,2.71,1.63
117,Ireland,194.34,7.76,2025Q2,172.80,5.15,2.61,1.67
118,Ireland,199.36,7.47,2025Q3,174.27,4.31,3.16,2.07
119,Ireland,203.23,6.98,2025Q4,175.80,3.49,3.49,2.87
120,Ireland,204.94,6.83,2026Q1,NaN,NaN,NaN,NaN
121,Ireland,205.94,5.97,2026Q2,NaN,NaN,NaN,NaN
236,Spain,163.12,8.26,2024Q3,111.24,2.22,6.04,2.33
237,Spain,166.07,11.36,2024Q4,111.86,2.37,8.99,2.33


In [51]:
# ---------------------------------------------------------
# CALCULATE REAL HOUSE-PRICE GROWTH
# ---------------------------------------------------------
#
# Nominal house-price growth doesn't account for inflation.
#
# For our screening tool we use the simple approximation:
#
#     Real House-Price Growth
#             =
#     House Price YoY Growth - Inflation
#
#
# Example:
#
# House prices       +7.0%
# Inflation          +2.5%
#                    -----
# Real growth        +4.5%
#
#
# NOTE:
# This is an analytical approximation rather than a formal
# inflation-adjusted house-price index.
#
# We'll describe it accurately in the application rather
# than implying greater precision than the metric provides.
# ---------------------------------------------------------


market_data["real_house_price_growth_pct"] = (
    market_data["house_price_yoy_pct"]
    - market_data["inflation_pct"]
)


market_data["real_house_price_growth_pct"] = (
    market_data["real_house_price_growth_pct"]
    .round(2)
)

In [52]:
# Display the latest observations where both property
# growth and inflation are available.

market_data[
    [
        "country",
        "quarter",
        "house_price_yoy_pct",
        "inflation_pct",
        "real_house_price_growth_pct"
    ]
].dropna().groupby("country").tail(6)

,country,quarter,house_price_yoy_pct,inflation_pct,real_house_price_growth_pct
114,Ireland,2024Q3,9.90,0.87,9.03
115,Ireland,2024Q4,9.37,0.53,8.84
116,Ireland,2025Q1,7.95,1.63,6.32
117,Ireland,2025Q2,7.76,1.67,6.09
118,Ireland,2025Q3,7.47,2.07,5.40
119,Ireland,2025Q4,6.98,2.87,4.11
236,Spain,2024Q3,8.26,2.33,5.93
237,Spain,2024Q4,11.36,2.33,9.03
238,Spain,2025Q1,12.29,2.67,9.62
239,Spain,2025Q2,12.77,2.17,10.60


In [53]:
# ---------------------------------------------------------
# DOWNLOAD UNEMPLOYMENT DATA
# ---------------------------------------------------------
#
# Source:
# Eurostat quarterly unemployment statistics.
#
# Why include unemployment?
#
# Labour-market weakness can affect:
#
#   - household affordability
#   - rental demand
#   - mortgage stress
#   - housing demand
#
# We use the total unemployment rate for the overall
# working-age population.
# ---------------------------------------------------------


unemployment_url = (
    "https://ec.europa.eu/eurostat/api/dissemination/"
    "statistics/1.0/data/une_rt_q"
)


unemployment_params = {

    # Ireland and Spain
    "geo": ["IE", "ES"],

    # Percentage of active population
    "unit": "PC_ACT",

    # Total population
    "sex": "T",

    # Ages 15-74
    "age": "Y15-74",

    # Seasonally adjusted
    "s_adj": "SA"
}


unemployment_response = requests.get(
    unemployment_url,
    params=unemployment_params,
    timeout=30
)


print(
    "Status:",
    unemployment_response.status_code
)

Status: 200


In [54]:
# Convert response to Python dictionary.

unemployment_data = unemployment_response.json()


# Always inspect external data before transforming it.
#
# This gives us a basic validation step and helps catch
# changes to the source dataset.

print(unemployment_data["label"])
print(unemployment_data["id"])
print(unemployment_data["size"])

Unemployment by sex and age - quarterly data
['freq', 's_adj', 'age', 'unit', 'sex', 'geo', 'time']
[1, 1, 1, 1, 1, 2, 94]


In [55]:
# ---------------------------------------------------------
# REUSABLE EUROSTAT JSON-STAT CONVERTER
# ---------------------------------------------------------
#
# We've now used the same conversion process several times.
#
# Instead of duplicating that logic for every dataset,
# we'll create one reusable function.
#
# INPUT:
#     Eurostat JSON response
#
# OUTPUT:
#     Normal Pandas DataFrame
#
# This makes the data pipeline:
#
# Eurostat API
#      ↓
# JSON-stat
#      ↓
# eurostat_to_dataframe()
#      ↓
# Pandas DataFrame
#
# ---------------------------------------------------------


def eurostat_to_dataframe(dataset):
    """
    Convert a Eurostat JSON-stat response into a
    standard Pandas DataFrame.

    Parameters
    ----------
    dataset : dict
        Parsed JSON response returned by the Eurostat API.

    Returns
    -------
    pandas.DataFrame
        One row per Eurostat observation with dimensions
        represented as columns.
    """

    # Identify dimensions such as:
    #
    # geo, time, unit, age, sex, etc.
    dimension_ids = dataset["id"]


    # Store the ordered categories for each dimension.
    dimension_values = {}


    for dim in dimension_ids:

        index = (
            dataset["dimension"][dim]
            ["category"]["index"]
        )

        if isinstance(index, dict):

            ordered = sorted(
                index.items(),
                key=lambda x: x[1]
            )

            dimension_values[dim] = [
                item[0] for item in ordered
            ]

        else:

            dimension_values[dim] = index


    # Generate every possible combination of dimensions.
    combinations = list(
        product(
            *[
                dimension_values[dim]
                for dim in dimension_ids
            ]
        )
    )


    rows = []


    # Match every combination with its observation value.
    for position, combination in enumerate(
        combinations
    ):

        value = dataset.get(
            "value",
            {}
        ).get(str(position))

        row = dict(
            zip(
                dimension_ids,
                combination
            )
        )

        row["value"] = value

        rows.append(row)


    return pd.DataFrame(rows)

In [56]:
# Use our reusable function to convert the unemployment
# dataset.

unemployment_raw = eurostat_to_dataframe(
    unemployment_data
)


unemployment_raw.head()

,freq,s_adj,age,unit,sex,geo,time,value
0,Q,SA,Y15-74,PC_ACT,T,IE,2003-Q1,NaN
1,Q,SA,Y15-74,PC_ACT,T,IE,2003-Q2,NaN
2,Q,SA,Y15-74,PC_ACT,T,IE,2003-Q3,NaN
3,Q,SA,Y15-74,PC_ACT,T,IE,2003-Q4,NaN
4,Q,SA,Y15-74,PC_ACT,T,IE,2004-Q1,NaN


In [57]:
# ---------------------------------------------------------
# CLEAN UNEMPLOYMENT DATA
# ---------------------------------------------------------


# Keep only the fields required by the market model.
unemployment = unemployment_raw[
    [
        "geo",
        "time",
        "value"
    ]
].copy()


# Give the generic Eurostat value column a meaningful name.
unemployment = unemployment.rename(
    columns={
        "value": "unemployment_pct"
    }
)


# Convert country codes.
unemployment["country"] = (
    unemployment["geo"].map({
        "IE": "Ireland",
        "ES": "Spain"
    })
)


# Convert Eurostat's quarter text into a Pandas Period.
unemployment["quarter"] = pd.PeriodIndex(
    unemployment["time"],
    freq="Q"
)


# Keep final fields.
unemployment = unemployment[
    [
        "country",
        "quarter",
        "unemployment_pct"
    ]
]


unemployment.dropna().groupby("country").tail(8)

,country,quarter,unemployment_pct
86,Ireland,2024Q3,4.2
87,Ireland,2024Q4,4.4
88,Ireland,2025Q1,4.4
89,Ireland,2025Q2,4.7
90,Ireland,2025Q3,4.9
91,Ireland,2025Q4,4.7
92,Ireland,2026Q1,4.9
93,Ireland,2026Q2,5.0
180,Spain,2024Q3,11.3
181,Spain,2024Q4,10.8


In [58]:
# ---------------------------------------------------------
# ADD UNEMPLOYMENT TO MASTER MARKET DATA
# ---------------------------------------------------------


market_data = pd.merge(
    market_data,
    unemployment,
    on=["country", "quarter"],
    how="outer"
)


market_data = (
    market_data
    .sort_values(
        ["country", "quarter"]
    )
    .reset_index(drop=True)
)


market_data.groupby("country").tail(8)

,country,house_price_index,house_price_yoy_pct,quarter,rent_index,rent_yoy_pct,price_rent_divergence_pp,inflation_pct,real_house_price_growth_pct,unemployment_pct
114,Ireland,185.50,9.90,2024Q3,167.07,4.79,5.11,0.87,9.03,4.2
115,Ireland,189.97,9.37,2024Q4,169.87,5.18,4.19,0.53,8.84,4.4
116,Ireland,191.83,7.95,2025Q1,171.47,5.24,2.71,1.63,6.32,4.4
117,Ireland,194.34,7.76,2025Q2,172.80,5.15,2.61,1.67,6.09,4.7
118,Ireland,199.36,7.47,2025Q3,174.27,4.31,3.16,2.07,5.40,4.9
119,Ireland,203.23,6.98,2025Q4,175.80,3.49,3.49,2.87,4.11,4.7
120,Ireland,204.94,6.83,2026Q1,NaN,NaN,NaN,NaN,NaN,4.9
121,Ireland,205.94,5.97,2026Q2,NaN,NaN,NaN,NaN,NaN,5.0
236,Spain,163.12,8.26,2024Q3,111.24,2.22,6.04,2.33,5.93,11.3
237,Spain,166.07,11.36,2024Q4,111.86,2.37,8.99,2.33,9.03,10.8


In [59]:
# ---------------------------------------------------------
# CURRENT ANALYTICAL MODEL
# ---------------------------------------------------------
#
# Display only our most important indicators so we can
# check the overall dataset before adding more sources.
# ---------------------------------------------------------


market_data[
    [
        "country",
        "quarter",
        "house_price_yoy_pct",
        "rent_yoy_pct",
        "inflation_pct",
        "unemployment_pct",
        "price_rent_divergence_pp",
        "real_house_price_growth_pct"
    ]
].dropna(
    subset=[
        "house_price_yoy_pct",
        "rent_yoy_pct"
    ]
).groupby("country").tail(6)

,country,quarter,house_price_yoy_pct,rent_yoy_pct,inflation_pct,unemployment_pct,price_rent_divergence_pp,real_house_price_growth_pct
114,Ireland,2024Q3,9.90,4.79,0.87,4.2,5.11,9.03
115,Ireland,2024Q4,9.37,5.18,0.53,4.4,4.19,8.84
116,Ireland,2025Q1,7.95,5.24,1.63,4.4,2.71,6.32
117,Ireland,2025Q2,7.76,5.15,1.67,4.7,2.61,6.09
118,Ireland,2025Q3,7.47,4.31,2.07,4.9,3.16,5.40
119,Ireland,2025Q4,6.98,3.49,2.87,4.7,3.49,4.11
236,Spain,2024Q3,8.26,2.22,2.33,11.3,6.04,5.93
237,Spain,2024Q4,11.36,2.37,2.33,10.8,8.99,9.03
238,Spain,2025Q1,12.29,2.38,2.67,10.8,9.91,9.62
239,Spain,2025Q2,12.77,2.39,2.17,10.6,10.38,10.60


In [60]:
# ---------------------------------------------------------
# DOWNLOAD GDP GROWTH DATA
# ---------------------------------------------------------
#
# Source:
# Eurostat quarterly national accounts.
#
# Why GDP?
#
# GDP growth gives us a broad measure of economic momentum.
#
# For property-market analysis it provides useful context
# around:
#
#   - household demand
#   - employment
#   - business activity
#   - general economic conditions
#
# IMPORTANT:
# GDP alone does NOT tell us whether a property market is
# attractive. It is simply one contextual indicator.
# ---------------------------------------------------------

gdp_url = (
    "https://ec.europa.eu/eurostat/api/dissemination/"
    "statistics/1.0/data/namq_10_gdp"
)

gdp_params = {

    # Ireland and Spain
    "geo": ["IE", "ES"],

    # Percentage change compared with the same
    # quarter of the previous year.
    "unit": "CLV_PCH_SM",

    # Gross Domestic Product
    "na_item": "B1GQ",

    # Calendar and seasonally adjusted data
    "s_adj": "SCA"
}

gdp_response = requests.get(
    gdp_url,
    params=gdp_params,
    timeout=30
)

print(
    "Status:",
    gdp_response.status_code
)

Status: 200


In [61]:
# ---------------------------------------------------------
# INSPECT GDP RESPONSE
# ---------------------------------------------------------
#
# Before transforming any external dataset, we inspect its
# metadata and dimensions.
#
# This helps ensure that the API returned what we expected.
# ---------------------------------------------------------

gdp_data = gdp_response.json()

print(gdp_data["label"])
print(gdp_data["id"])
print(gdp_data["size"])

Gross domestic product (GDP) and main components (output, expenditure and income) - quarterly data
['freq', 'unit', 's_adj', 'na_item', 'geo', 'time']
[1, 1, 1, 1, 2, 206]


In [62]:
# ---------------------------------------------------------
# CONVERT GDP JSON INTO A DATAFRAME
# ---------------------------------------------------------
#
# We created eurostat_to_dataframe() earlier so we no
# longer need to repeat all the JSON-stat conversion code.
# ---------------------------------------------------------

gdp_raw = eurostat_to_dataframe(
    gdp_data
)

gdp_raw.head()

,freq,unit,s_adj,na_item,geo,time,value
0,Q,CLV_PCH_SM,SCA,B1GQ,IE,1975-Q1,NaN
1,Q,CLV_PCH_SM,SCA,B1GQ,IE,1975-Q2,NaN
2,Q,CLV_PCH_SM,SCA,B1GQ,IE,1975-Q3,NaN
3,Q,CLV_PCH_SM,SCA,B1GQ,IE,1975-Q4,NaN
4,Q,CLV_PCH_SM,SCA,B1GQ,IE,1976-Q1,NaN


In [63]:
# ---------------------------------------------------------
# CLEAN GDP DATA
# ---------------------------------------------------------

# Keep only country, time and observation value.
gdp = gdp_raw[
    [
        "geo",
        "time",
        "value"
    ]
].copy()


# Rename the generic value column.
gdp = gdp.rename(
    columns={
        "value": "gdp_growth_pct"
    }
)


# Convert country codes into readable names.
gdp["country"] = gdp["geo"].map({
    "IE": "Ireland",
    "ES": "Spain"
})


# Convert the Eurostat quarter field into our standard
# Pandas quarterly format.
gdp["quarter"] = pd.PeriodIndex(
    gdp["time"],
    freq="Q"
)


# Keep only final analytical fields.
gdp = gdp[
    [
        "country",
        "quarter",
        "gdp_growth_pct"
    ]
]


# Display latest observations.
gdp.dropna().groupby("country").tail(8)

,country,quarter,gdp_growth_pct
198,Ireland,2024Q3,4.9
199,Ireland,2024Q4,12.2
200,Ireland,2025Q1,16.6
201,Ireland,2025Q2,11.7
202,Ireland,2025Q3,5.1
203,Ireland,2025Q4,-0.1
204,Ireland,2026Q1,-13.4
205,Ireland,2026Q2,-0.4
404,Spain,2024Q3,3.9
405,Spain,2024Q4,3.7


In [ ]:
# ---------------------------------------------------------
# MERGE GDP INTO THE MARKET MODEL
# ---------------------------------------------------------

market_data = pd.merge(
    market_data,
    gdp,
    on=["country", "quarter"],
    how="outer"
)


market_data = (
    market_data
    .sort_values(
        ["country", "quarter"]
    )
    .reset_index(drop=True)
)


market_data.groupby("country").tail(8)

,country,house_price_index,house_price_yoy_pct,quarter,rent_index,rent_yoy_pct,price_rent_divergence_pp,inflation_pct,real_house_price_growth_pct,unemployment_pct,gdp_growth_pct
198,Ireland,185.50,9.90,2024Q3,167.07,4.79,5.11,0.87,9.03,4.2,4.9
199,Ireland,189.97,9.37,2024Q4,169.87,5.18,4.19,0.53,8.84,4.4,12.2
200,Ireland,191.83,7.95,2025Q1,171.47,5.24,2.71,1.63,6.32,4.4,16.6
201,Ireland,194.34,7.76,2025Q2,172.80,5.15,2.61,1.67,6.09,4.7,11.7
202,Ireland,199.36,7.47,2025Q3,174.27,4.31,3.16,2.07,5.40,4.9,5.1
203,Ireland,203.23,6.98,2025Q4,175.80,3.49,3.49,2.87,4.11,4.7,-0.1
204,Ireland,204.94,6.83,2026Q1,NaN,NaN,NaN,NaN,NaN,4.9,-13.4
205,Ireland,205.94,5.97,2026Q2,NaN,NaN,NaN,NaN,NaN,5.0,-0.4
404,Spain,163.12,8.26,2024Q3,111.24,2.22,6.04,2.33,5.93,11.3,3.9
405,Spain,166.07,11.36,2024Q4,111.86,2.37,8.99,2.33,9.03,10.8,3.7


In [65]:
# ---------------------------------------------------------
# ECB DEPOSIT FACILITY RATE
# ---------------------------------------------------------
#
# Source:
# European Central Bank
#
# Series:
# FM.D.U2.EUR.4F.KR.DFR.LEV
#
# We use the ECB Deposit Facility Rate because the ECB
# currently uses this rate to steer its monetary-policy
# stance.
#
# Unlike our earlier prototype values, the observations
# below are sourced from the ECB's official interest-rate
# history.
#
# The rate changes on specific dates rather than quarterly.
# We therefore store the official effective dates first and
# later convert them into quarterly observations.
# ---------------------------------------------------------

ecb_rate_changes = pd.DataFrame({

    "date": pd.to_datetime([
        "2023-02-08",
        "2023-03-22",
        "2023-05-10",
        "2023-06-21",
        "2023-08-02",
        "2023-09-20",

        "2024-06-12",
        "2024-09-18",
        "2024-10-23",
        "2024-12-18",

        "2025-02-05",
        "2025-03-12",
        "2025-04-23",
        "2025-06-11"
    ]),

    "policy_rate_pct": [
        2.50,
        3.00,
        3.25,
        3.50,
        3.75,
        4.00,

        3.75,
        3.50,
        3.25,
        3.00,

        2.75,
        2.50,
        2.25,
        2.00
    ]
})


ecb_rate_changes

,date,policy_rate_pct
0,2023-02-08,2.50
1,2023-03-22,3.00
2,2023-05-10,3.25
3,2023-06-21,3.50
4,2023-08-02,3.75
5,2023-09-20,4.00
6,2024-06-12,3.75
7,2024-09-18,3.50
8,2024-10-23,3.25
9,2024-12-18,3.00


In [66]:
# ---------------------------------------------------------
# CONVERT ECB RATE CHANGES TO QUARTERLY OBSERVATIONS
# ---------------------------------------------------------
#
# ECB policy rates change on specific dates.
#
# Our property-market model is quarterly, so we need to
# align the policy-rate series with that frequency.
#
# For this prototype we use the rate prevailing at the END
# of each quarter.
#
# This gives us a simple and reproducible financing
# indicator that aligns with the rest of our market model.
# ---------------------------------------------------------


# Create a daily time series covering our analytical period.
daily_dates = pd.DataFrame({
    "date": pd.date_range(
        start=ecb_rate_changes["date"].min(),
        end="2025-12-31",
        freq="D"
    )
})


# Join each date to the most recent ECB rate decision.
#
# merge_asof() effectively means:
#
# "For this date, find the latest policy rate that had
# already become effective."
daily_rates = pd.merge_asof(
    daily_dates,
    ecb_rate_changes.sort_values("date"),
    on="date",
    direction="backward"
)


# Assign every day to a calendar quarter.
daily_rates["quarter"] = (
    daily_rates["date"]
    .dt
    .to_period("Q")
)


# Take the final observation from each quarter.
#
# This represents the ECB Deposit Facility Rate prevailing
# at quarter end.
ecb_rates = (
    daily_rates
    .groupby(
        "quarter",
        as_index=False
    )
    .tail(1)
)


# Keep only the fields required by our market model.
ecb_rates = ecb_rates[
    [
        "quarter",
        "policy_rate_pct"
    ]
].reset_index(drop=True)


ecb_rates

,quarter,policy_rate_pct
0,2023Q1,3.00
1,2023Q2,3.50
2,2023Q3,4.00
3,2023Q4,4.00
4,2024Q1,4.00
5,2024Q2,3.75
6,2024Q3,3.50
7,2024Q4,3.00
8,2025Q1,2.50
9,2025Q2,2.00


In [67]:
# ---------------------------------------------------------
# ADD ECB POLICY RATE
# ---------------------------------------------------------
#
# Unlike the other indicators, the ECB rate is shared
# across Ireland and Spain.
#
# Therefore:
#
#     Ireland 2024Q1 → ECB rate
#     Spain   2024Q1 → same ECB rate
#
# We merge using quarter only.
# ---------------------------------------------------------

market_data = pd.merge(
    market_data,
    ecb_rates,
    on="quarter",
    how="left"
)


market_data = (
    market_data
    .sort_values(
        ["country", "quarter"]
    )
    .reset_index(drop=True)
)

In [68]:
# ---------------------------------------------------------
# DISPLAY COMPLETE MARKET MODEL
# ---------------------------------------------------------

display_columns = [
    "country",
    "quarter",
    "house_price_yoy_pct",
    "rent_yoy_pct",
    "inflation_pct",
    "unemployment_pct",
    "gdp_growth_pct",
    "policy_rate_pct",
    "price_rent_divergence_pp",
    "real_house_price_growth_pct"
]


market_data[
    display_columns
].groupby("country").tail(8)

,country,quarter,house_price_yoy_pct,rent_yoy_pct,inflation_pct,unemployment_pct,gdp_growth_pct,policy_rate_pct,price_rent_divergence_pp,real_house_price_growth_pct
198,Ireland,2024Q3,9.90,4.79,0.87,4.2,4.9,3.5,5.11,9.03
199,Ireland,2024Q4,9.37,5.18,0.53,4.4,12.2,3.0,4.19,8.84
200,Ireland,2025Q1,7.95,5.24,1.63,4.4,16.6,2.5,2.71,6.32
201,Ireland,2025Q2,7.76,5.15,1.67,4.7,11.7,2.0,2.61,6.09
202,Ireland,2025Q3,7.47,4.31,2.07,4.9,5.1,2.0,3.16,5.40
203,Ireland,2025Q4,6.98,3.49,2.87,4.7,-0.1,2.0,3.49,4.11
204,Ireland,2026Q1,6.83,NaN,NaN,4.9,-13.4,NaN,NaN,NaN
205,Ireland,2026Q2,5.97,NaN,NaN,5.0,-0.4,NaN,NaN,NaN
404,Spain,2024Q3,8.26,2.22,2.33,11.3,3.9,3.5,6.04,5.93
405,Spain,2024Q4,11.36,2.37,2.33,10.8,3.7,3.0,8.99,9.03


In [69]:
# ---------------------------------------------------------
# COMPLETE SOURCE PROVENANCE TABLE
# ---------------------------------------------------------
#
# The application uses two types of external evidence:
#
# 1. Quantitative sources
#    Used to establish numerical market observations.
#
# 2. Contextual sources
#    Used by the AI to understand policy, regulation and
#    structural market conditions.
#
# Keeping these separate helps us explain exactly where
# each type of information comes from.
# ---------------------------------------------------------

data_sources = pd.DataFrame([

    {
        "type": "Quantitative",
        "market": "Ireland / Spain",
        "indicator": "House Prices",
        "provider": "Eurostat",
        "source": "prc_hpi_q"
    },

    {
        "type": "Quantitative",
        "market": "Ireland / Spain",
        "indicator": "Rent",
        "provider": "Eurostat",
        "source": "prc_hicp_midx"
    },

    {
        "type": "Quantitative",
        "market": "Ireland / Spain",
        "indicator": "Inflation",
        "provider": "Eurostat",
        "source": "prc_hicp_manr"
    },

    {
        "type": "Quantitative",
        "market": "Ireland / Spain",
        "indicator": "Unemployment",
        "provider": "Eurostat",
        "source": "une_rt_q"
    },

    {
        "type": "Quantitative",
        "market": "Ireland / Spain",
        "indicator": "GDP Growth",
        "provider": "Eurostat",
        "source": "namq_10_gdp"
    },

    {
        "type": "Quantitative",
        "market": "Euro Area",
        "indicator": "Policy Rate",
        "provider": "European Central Bank",
        "source": "ECB policy-rate series"
    },

    {
        "type": "Contextual",
        "market": "Ireland",
        "indicator": "Housing Policy / Supply",
        "provider": "Government of Ireland",
        "source": "Delivering Homes, Building Communities 2025-2030"
    },

    {
        "type": "Contextual",
        "market": "Ireland",
        "indicator": "Housing Supply / Economic Context",
        "provider": "Central Bank of Ireland",
        "source": "Quarterly Bulletin Q3 2026"
    },

    {
        "type": "Contextual",
        "market": "Spain",
        "indicator": "Housing Market / Supply",
        "provider": "Banco de España",
        "source": "Annual Report 2025"
    },

    {
        "type": "Contextual",
        "market": "Spain",
        "indicator": "Rental Regulation",
        "provider": "Boletín Oficial del Estado",
        "source": "Law 12/2023 on the Right to Housing"
    }

])


# Save the updated provenance table.
data_sources.to_csv(
    "data/data_sources.csv",
    index=False
)


data_sources

,type,market,indicator,provider,source
0,Quantitative,Ireland / Spain,House Prices,Eurostat,prc_hpi_q
1,Quantitative,Ireland / Spain,Rent,Eurostat,prc_hicp_midx
2,Quantitative,Ireland / Spain,Inflation,Eurostat,prc_hicp_manr
3,Quantitative,Ireland / Spain,Unemployment,Eurostat,une_rt_q
4,Quantitative,Ireland / Spain,GDP Growth,Eurostat,namq_10_gdp
5,Quantitative,Euro Area,Policy Rate,European Central Bank,ECB policy-rate series
6,Contextual,Ireland,Housing Policy / Supply,Government of Ireland,"Delivering Homes, Building Communities 2025-2030"
7,Contextual,Ireland,Housing Supply / Economic Context,Central Bank of Ireland,Quarterly Bulletin Q3 2026
8,Contextual,Spain,Housing Market / Supply,Banco de España,Annual Report 2025
9,Contextual,Spain,Rental Regulation,Boletín Oficial del Estado,Law 12/2023 on the Right to Housing


In [74]:
# ---------------------------------------------------------
# SAVE THE ANALYTICAL DATASET
# ---------------------------------------------------------
#
# Streamlit should not have to call five external APIs
# every time we run the application.
#
# For the interview prototype we cache the cleaned,
# validated analytical dataset locally.
#
# Benefits:
#
#   - faster application
#   - reproducible demo
#   - no dependency on Eurostat being available
#   - easier debugging
#
# In production, this could be replaced by a scheduled
# ingestion pipeline.
# ---------------------------------------------------------

from pathlib import Path


# Create the data directory if it doesn't already exist.
Path("data").mkdir(
    exist_ok=True
)


# CSV cannot preserve Pandas Period objects particularly
# well, so convert quarter to text before saving.
market_export = market_data.copy()

market_export["quarter"] = (
    market_export["quarter"]
    .astype(str)
)


# Save market observations.
market_export.to_csv(
    "data/market_data.csv",
    index=False
)


# Save source metadata separately.
data_sources.to_csv(
    "data/data_sources.csv",
    index=False
)


print("Files saved successfully.")

Files saved successfully.


In [75]:
# ---------------------------------------------------------
# VERIFY SAVED FILE
# ---------------------------------------------------------
#
# Reload the CSV exactly as Streamlit will later load it.
#
# This catches export/import problems before we start
# building the application.
# ---------------------------------------------------------

test_data = pd.read_csv(
    "data/market_data.csv"
)


print(
    "Rows:",
    len(test_data)
)

print(
    "Countries:",
    test_data["country"].unique()
)

print(
    "Columns:"
)

print(
    test_data.columns.tolist()
)


test_data.tail()

Rows: 412
Countries: <ArrowStringArray>
['Ireland', 'Spain']
Length: 2, dtype: str
Columns:
['country', 'house_price_index', 'house_price_yoy_pct', 'quarter', 'rent_index', 'rent_yoy_pct', 'price_rent_divergence_pp', 'inflation_pct', 'real_house_price_growth_pct', 'unemployment_pct', 'gdp_growth_pct', 'policy_rate_pct']


,country,house_price_index,house_price_yoy_pct,quarter,rent_index,rent_yoy_pct,price_rent_divergence_pp,inflation_pct,real_house_price_growth_pct,unemployment_pct,gdp_growth_pct,policy_rate_pct
407,Spain,178.89,12.77,2025Q2,113.37,2.39,10.38,2.17,10.60,10.6,2.6,2.0
408,Spain,184.06,12.84,2025Q3,113.95,2.44,10.40,2.80,10.04,10.5,2.4,2.0
409,Spain,187.45,12.87,2025Q4,114.58,2.43,10.44,3.13,9.74,10.2,2.5,2.0
410,Spain,194.06,12.83,2026Q1,NaN,NaN,NaN,NaN,NaN,10.3,2.6,NaN
411,Spain,200.60,12.14,2026Q2,NaN,NaN,NaN,NaN,NaN,10.2,2.6,NaN


In [72]:
# Check that we have only one policy-rate column
[col for col in market_data.columns if "policy_rate" in col]

['policy_rate_pct']

In [73]:
market_data[
    ["country", "quarter", "policy_rate_pct"]
].tail(20)

,country,quarter,policy_rate_pct
392,Spain,2021Q3,NaN
393,Spain,2021Q4,NaN
394,Spain,2022Q1,NaN
395,Spain,2022Q2,NaN
396,Spain,2022Q3,NaN
397,Spain,2022Q4,NaN
398,Spain,2023Q1,3.00
399,Spain,2023Q2,3.50
400,Spain,2023Q3,4.00
401,Spain,2023Q4,4.00


In [76]:
# Check the latest observations being supplied to the AI.
market_data[
    [
        "country",
        "quarter",
        "gdp_growth_pct",
        "house_price_yoy_pct",
        "rent_yoy_pct",
        "inflation_pct",
        "unemployment_pct",
        "policy_rate_pct"
    ]
].groupby("country").tail(4)

,country,quarter,gdp_growth_pct,house_price_yoy_pct,rent_yoy_pct,inflation_pct,unemployment_pct,policy_rate_pct
202,Ireland,2025Q3,5.1,7.47,4.31,2.07,4.9,2.0
203,Ireland,2025Q4,-0.1,6.98,3.49,2.87,4.7,2.0
204,Ireland,2026Q1,-13.4,6.83,NaN,NaN,4.9,NaN
205,Ireland,2026Q2,-0.4,5.97,NaN,NaN,5.0,NaN
408,Spain,2025Q3,2.4,12.84,2.44,2.80,10.5,2.0
409,Spain,2025Q4,2.5,12.87,2.43,3.13,10.2,2.0
410,Spain,2026Q1,2.6,12.83,NaN,NaN,10.3,NaN
411,Spain,2026Q2,2.6,12.14,NaN,NaN,10.2,NaN
